In [ ]:
import json
import time
import requests
import pandas as pd
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

In [ ]:
multiclass_dataset_path = kagglehub.dataset_download("sagarikashreevastava/cognitive-distortion-detetction-dataset")
print("Path to dataset files:", multiclass_dataset_path)
multiclass_dataset_file_path = multiclass_dataset_path + "/Annotated_data.csv"
df = pd.read_csv(multiclass_dataset_file_path) 
df = df.drop('Id_Number', axis=1) # delete columnb with id 
df

Path to dataset files: C:\Users\kzvau\.cache\kagglehub\datasets\sagarikashreevastava\cognitive-distortion-detetction-dataset\versions\1


,Patient Question,Distorted part,Dominant Distortion,Secondary Distortion (Optional)
0,"Hello, I have a beautiful,smart,outgoing and a...",The voice are always fimilar (someone she know...,Personalization,NaN
1,Since I was about 16 years old I’ve had these ...,I feel trapped inside my disgusting self and l...,Labeling,Emotional Reasoning
2,So I’ve been dating on and off this guy for a...,NaN,No Distortion,NaN
3,My parents got divorced in 2004. My mother has...,NaN,No Distortion,NaN
4,I don’t really know how to explain the situati...,I refused to go because I didn’t know if it wa...,Fortune-telling,Emotional Reasoning
...,...,...,...,...
2525,I’m a 21 year old female. I spent most of my l...,NaN,No Distortion,NaN
2526,I am 21 female and have not had any friends fo...,Now I am at university my peers around me all ...,Overgeneralization,NaN
2527,From the U.S.: My brother is 19 years old and ...,He claims he’s severely depressed and has outb...,Mental filter,Mind Reading
2528,From the U.S.: I am a 21 year old woman who ha...,NaN,No Distortion,NaN


In [3]:
def choose_text(row):
    if pd.notna(row["Distorted part"]):
        return row["Distorted part"]
    return row["Patient Question"]

def collect_labels(row):
    labels = []
    dominant = row["Dominant Distortion"]
    if dominant != "No Distortion":
        labels.append(dominant)
    secondary = row["Secondary Distortion (Optional)"]
    if pd.notna(secondary):
        if secondary != "No Distortion" and secondary not in labels:
            labels.append(secondary)
    return labels

data = pd.DataFrame()

data["text"] = df.apply(choose_text, axis=1)
data["labels"] = df.apply(collect_labels, axis=1)

data["dominant_label"] = df["Dominant Distortion"]

all_labels = sorted({
    label
    for labels in data["labels"]
    for label in labels
})

data

,text,labels,dominant_label
0,The voice are always fimilar (someone she know...,[Personalization],Personalization
1,I feel trapped inside my disgusting self and l...,"[Labeling, Emotional Reasoning]",Labeling
2,So I’ve been dating on and off this guy for a...,[],No Distortion
3,My parents got divorced in 2004. My mother has...,[],No Distortion
4,I refused to go because I didn’t know if it wa...,"[Fortune-telling, Emotional Reasoning]",Fortune-telling
...,...,...,...
2525,I’m a 21 year old female. I spent most of my l...,[],No Distortion
2526,Now I am at university my peers around me all ...,[Overgeneralization],Overgeneralization
2527,He claims he’s severely depressed and has outb...,"[Mental filter, Mind Reading]",Mental filter
2528,From the U.S.: I am a 21 year old woman who ha...,[],No Distortion


In [4]:
SEED = 42

train_df, test_df = train_test_split(
    data,
    test_size=0.20,
    random_state=SEED,
    shuffle=True,
    stratify=data["dominant_label"]
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train size:", len(train_df))
print("Test size:", len(test_df))

Train size: 2024
Test size: 506


In [5]:
def labels_to_json(labels):
    return json.dumps(labels, ensure_ascii=False)

def build_many_shot_prefix(train_df, allowed_labels):
    parts = []
    parts.append(
        "You are a professional psychotherapist experienced in cognitive-behavioral therapy.\n"
        "Your task is multi-label classification of cognitive distortions in patient texts.\n"
        "You can assign none, one, or several cognitive distortion labels to each text.\n"
        "Allowed labels are:")

    parts.append(labels_to_json(allowed_labels))
    
    parts.append(
        "Important annotation rule: if there is no cognitive distortion, return an empty JSON array: [].\n"
        "Do not use the label 'No Distortion'. Absence of distortion must be represented only as [].\n"
        "Return only a valid JSON array of strings. Do not return explanations, comments, markdown, or extra text.\n"
        "Below are annotated examples."
    )

    for i, row in train_df.iterrows():
        parts.append(f"\nExample {i + 1}:")
        parts.append(f"Text: {row['text']}")
        parts.append(f"Labels: {labels_to_json(row['labels'])}")

    parts.append(
        "\nNow classify new texts using the same label set and the same annotation logic.")
    return "\n".join(parts)

many_shot_prefix = build_many_shot_prefix(train_df, all_labels)

In [6]:
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "qwen3.5:4b"

NUM_CTX = 262144

response_schema = {
    "type": "array",
    "items": {
        "type": "string",
        "enum": all_labels
    }
}

In [7]:
def build_test_prompt(many_shot_prefix, text):
    return (
        many_shot_prefix
        + "\n\nText to classify:\n"
        + text
        + "\n\nReturn only the JSON array of labels:")

def call_ollama(prompt, num_ctx=NUM_CTX):
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL_NAME,
            "prompt": prompt,
            "think": False,
            "stream": False,
            "format": response_schema,
            "options": {
                "temperature": 0,
                "num_ctx": num_ctx,
                "num_predict": 128
            }
        },
        timeout=3600)
    response.raise_for_status()
    return response.json()

def parse_model_answer(raw_answer, allowed_labels):
    try:
        parsed = json.loads(raw_answer)
    except json.JSONDecodeError:
        return None

    if not isinstance(parsed, list):
        return None

    cleaned = []

    for label in parsed:
        if not isinstance(label, str):
            continue
        label = label.strip()
        if label in allowed_labels and label not in cleaned:
            cleaned.append(label)

    return cleaned

In [8]:
def calculate_macro_f1(results_df, all_labels):
    
    valid_results_df = results_df[
        results_df["predicted_labels"].notna()
    ].copy()

    if len(valid_results_df) == 0:
        return None

    mlb = MultiLabelBinarizer(classes=all_labels)

    y_true = mlb.fit_transform(valid_results_df["true_labels"])
    y_pred = mlb.transform(valid_results_df["predicted_labels"])

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    return macro_f1

In [22]:
def run_many_shot_experiment(test_df, limit=None):
    if limit is None:
        test_part = test_df.copy()
    else:
        test_part = test_df.head(limit).copy()

    results = []

    experiment_start_time = time.time()

    for position, row in test_part.iterrows():
        text = row["text"]
        true_labels = row["labels"]

        prompt = build_test_prompt(many_shot_prefix, text)

        start_time = time.time()

        result = call_ollama(prompt)
        raw_answer = result["response"]
        predicted_labels = parse_model_answer(raw_answer, all_labels)

        text_time_sec = time.time() - start_time

        results.append({
            #"index": position,
            "text": text,
            "true_labels": true_labels,
            # "raw_answer": raw_answer,
            "predicted_labels": predicted_labels,
            # "error": error,
            # "text_time_sec": text_time_sec
        })

    total_time_sec = time.time() - experiment_start_time

    results_df = pd.DataFrame(results)

    macro_f1 = calculate_macro_f1(
        results_df=results_df,
        all_labels=all_labels
    )

    metrics_df = pd.DataFrame([{
        "macro_f1": macro_f1,
        "total_time_sec": total_time_sec
    }])

    print("Total time, sec:", round(total_time_sec, 2))
    print("Macro F1:", macro_f1)

    return results_df, metrics_df

In [23]:
one_text_results_df, one_text_metrics_df = run_many_shot_experiment(
    test_df=test_df,
    limit=1
)

display(one_text_results_df)

Total time, sec: 333.37
Macro F1: 0.0


,text,true_labels,predicted_labels
0,From a 14 year old in the U.S.: Hi so I’m kind...,[],"[All-or-nothing thinking, Fortune-telling]"


In [24]:
three_texts_results_df, three_texts_metrics_df = run_many_shot_experiment(
    test_df=test_df,
    limit=3
)

display(three_texts_results_df)

Total time, sec: 38.07
Macro F1: 0.1


,text,true_labels,predicted_labels
0,From a 14 year old in the U.S.: Hi so I’m kind...,[],"[All-or-nothing thinking, Fortune-telling]"
1,I’m good at hiding as everyone perceives me to...,[Mind Reading],"[Labeling, Mind Reading]"
2,My husband’s father committed murder/suicide l...,[],[]


In [ ]:
"""
all_texts_results_df, all_texts_metrics_df = run_many_shot_experiment(
    test_df=test_df,
    limit=None
)

display(all_texts_results_df.head())
"""

'\nall_texts_results_df, all_texts_metrics_df = run_many_shot_experiment(\n    test_df=test_df,\n    limit=None\n)\n\ndisplay(all_texts_metrics_df)\ndisplay(all_texts_results_df.head())\n'